# Multimodal Music Analysis

The target of this project is the implementation of a complete pipeline for extracting knowledge from music data by combining two different modalities: **Audio** and **Text**. The project is split into two parts:  
1. Preprocessing of the data, generation of embeddings and researching analysis.
2. Classification, Fusion and Evaluation.  
  
This notebook contains the implementation of part 2.

**Contributors**

|         Name         |          AM          |
| -------------------- | -------------------- |
| Διονύσιος Κοτσίρης   |   1115201900089      |
| Ιωάννης Οικονομίδης  |   1115201600120      |

Requirements:

In [ ]:
%pip install seaborn
%pip install datasets
%pip install matplotlib
%pip install vaderSentiment
%pip install kneed
%pip install contractions
%pip install pyspellchecker
%pip install scikit-learn
%pip install pandas
%pip install wordcloud
%pip install tensorflow
%pip install gensim
%pip install nltk

## 1. Classification Models

In [1]:
import pandas as pd
import os
import re

import numpy as np
import tensorflow as tf
import sklearn

import matplotlib.pyplot as plt

Firstly, we load the collected data from the part 1.
- The clean dataframe with the songs, genres, lyrics and MFCC stats.
- The audio and text embeddings  

In the first part, we completed a small case study by comparing the Autoencoder's and PCA's audio embeddings. In this part, we choose to work with the autoencoder's embeddings, because of their reduced dimensionality and high quality.

In [4]:
# Loading the data and the audio embeddings created in part 1.
final_df_path = "data/final_data.pkl"
if (os.path.isfile(path=final_df_path)):
    data = pd.read_pickle(final_df_path,)

data.head()

,id,genres,main_genre,song,tags,audio_stats,lyrics
0,006TYKNjNxWjfKjy,"[symphonic metal, symphonic power metal, power...",metal,Flames of Revenge,"symphonic metal,power metal,symphonic power metal","[23.30799674987793, -8.152815818786621, -17.38...",princess must assist death endur tragic pain c...
1,00i0OB17z3m3yamO,"[alternative metal, alternative rock, rock]",metal,Sign of the Times,"alternative rock,rock,alternative metal,altern...","[24.25779724121093, -10.212427139282228, -4.85...",sign time anoth mountain climb sun burn hot fl...
2,00mLjKFmaTv9MdaB,"[rap, hip hop, pop]",rap,Boom Boom Pow,"dance,hip hop,pop,hip-hop,rap","[20.77475357055664, -6.112589359283447, -10.24...",get get get get get get get get boom boom boom...
3,01ScM29f0wF2Mnkq,"[rap, indie pop, electronic]",rap,Implicit Demand for Proof,"electronic,alternative,rap,indie,indie pop","[21.80247497558593, 2.8943941593170166, -16.29...",know liar know could set fire day go ahead mak...
4,01gyRHLquwXDlhkO,[rap],rap,Somebody's Gotta Die,"rap,hip-hop,gangsta rap,east coast rap","[22.731481552124023, -12.233457565307615, 2.38...",sittin crib dreamin leer jet coup way salt sho...


In [3]:
audio_embeddings = np.load("data/audio_embeddings.npy")
audio_embeddings.shape

(17299, 32)

In [5]:
text_embeddings = np.load("data/text_embeddings.npy")
text_embeddings.shape

(17299, 100)

Next step, we set up the environment for the unimodal classification.
- exctracting the data and the labels
- choose and train classifier  
  
Since the embeddings are already fixed-size vector representations, classical classifiers such as SVM and Random Forest are more appropriate than RNNs or Transformers.  
  
We will train two different models (SVM and Random Forest classifier) and we will keep the one with the better metrics.

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, Normalizer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score


In [41]:
'''
Function that trains the classifier
Args:
    - x_data: the training data
    - y_labels: labels corresponding to the data
    - available models: 
        - "svm" : SVM
        - "rf"  : Random Forest
'''
def genre_classifier(x_data, y_labels, model_name="svm"):

    le = LabelEncoder()
    y_encoded = le.fit_transform(y_labels)

    X_train, X_test, y_train, y_test = train_test_split(
        x_data,
        y_encoded,
        test_size=0.2,
        random_state=42,
        stratify=y_encoded
    )

    if model_name == "rf":
        model = RandomForestClassifier(
            n_estimators=800,
            max_depth=None,
            random_state=42,
            class_weight="balanced"
        )

    elif model_name == "svm":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", C=10, gamma="scale", class_weight=None))
        ])

    else:
        print("Model is needed")
        return
    
    # training
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print(f"{model_name} Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print(f"{model_name} Macro F1: {f1_score(y_test, y_pred, average='macro'):.2f}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    return model




### Model B: audio classifier

In [37]:
''' SVM Classifier and Results '''
svm_model = genre_classifier(audio_embeddings, data["main_genre"], model_name="svm")


svm Accuracy: 0.86
svm Macro F1: 0.70
              precision    recall  f1-score   support

   classical       0.69      0.55      0.62       123
        jazz       0.69      0.56      0.61       305
       latin       0.56      0.42      0.48       177
       metal       0.90      0.95      0.93      2060
         rap       0.86      0.86      0.86       795

    accuracy                           0.86      3460
   macro avg       0.74      0.67      0.70      3460
weighted avg       0.85      0.86      0.85      3460



In [42]:
''' Random Forest Classifier and Results '''
rf_model = genre_classifier(audio_embeddings, data["main_genre"], model_name="rf")

rf Accuracy: 0.82
rf Macro F1: 0.59
              precision    recall  f1-score   support

   classical       0.74      0.26      0.39       123
        jazz       0.66      0.41      0.51       305
       latin       0.67      0.21      0.32       177
       metal       0.87      0.95      0.91      2060
         rap       0.76      0.88      0.82       795

    accuracy                           0.82      3460
   macro avg       0.74      0.54      0.59      3460
weighted avg       0.81      0.82      0.80      3460



SVM > Random Forest

Ablations:
- "C": [0.1, 1, 10, 50],
- "kernel": ["rbf", "linear"],
- "gamma": ["scale", 0.01, 0.001],
- "class_weight": [None, "balanced"] 